# 🚀 Notebook 06 — Deployment with Gradio

| Field | Details |
|---|---|
| Input | `models/transformer_ner/` (best model) |
| Output | Interactive Gradio Web App |
| Runtime | ~1 min to start |
| Depends On | Notebook 04 (Transformer model) |
| This is the final notebook |

## 1. Setup & Imports

### 💡 The Concept of ML Deployment
Deployment is the final stage of the machine learning lifecycle. It's the process of moving a model from research/development to a production environment where it can be used by end-users or integrated into other software systems. Without deployment, a model is just an academic exercise. 

Here, we will bring our NER model to life with an interactive web application.

In [ ]:
# 📦 Import required libraries
import gradio as gr
from transformers import pipeline
import os

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries loaded successfully.")

## 2. Why Deploy?

### 🌉 Bridging the Gap
The gap between a trained model (a collection of weights and biases in an `.h5`, `.pt`, or `.bin` file) and a usable product is significant. A non-technical stakeholder cannot interact with a Jupyter Notebook or a raw API easily. Deployment provides an intuitive interface.

### 🛠️ Why Gradio?
Gradio is an open-source Python library that allows you to quickly build customizable UI components around your machine learning models. 
* **Gradio**: Ideal for ML demos, prototyping, and quick sharing. Extremely fast to set up with its `Blocks` and `Interface` APIs.
* **Streamlit**: Great for data dashboards and more complex multi-page apps.
* **Flask/FastAPI**: The standard for building robust REST APIs for backend integration.

We will use Gradio to build a beautiful, interactive web app and later discuss how to host it for free on **HuggingFace Spaces**.

## 3. Load Best Model

We use the Transformer (DistilBERT/BERT) model as it achieved the highest F1 score in our evaluations (Notebook 04).

We will attempt to load the local fine-tuned model from `../models/transformer_ner/`. If the model is not found locally (e.g., if this notebook is run in a fresh environment), we will gracefully fallback to downloading `dslim/bert-base-NER` from the HuggingFace Hub.

In [ ]:
# 🔍 Try to load local model, otherwise fallback
local_model_path = "../models/transformer_ner/"

if os.path.exists(local_model_path):
    print(f"📦 Loading local model from: {local_model_path}")
    model_id = local_model_path
else:
    model_id = "dslim/bert-base-NER"
    print(f"⚠️ Local model not found. Loading fallback from Hub: {model_id}")

# Initialize the pipeline
# aggregation_strategy='simple' automatically merges B- and I- tokens into a single entity
ner_pipeline = pipeline("ner", model=model_id, aggregation_strategy="simple")

print("✅ Model pipeline ready.")

## 4. Build Gradio Interface

We will use the **Gradio Blocks API**, which offers more flexibility than the standard `Interface` API. It allows us to arrange components in rows and columns.

We will use the `HighlightedText` component, which is specifically designed to visualize text spans with associated labels—perfect for NER!

We will map entity types to specific colors for visual clarity:
- **PER (Person)**: Red
- **ORG (Organization)**: Blue
- **LOC (Location)**: Green
- **MISC (Miscellaneous)**: Yellow

In [ ]:
# 🎨 Define the prediction function for Gradio
def predict_entities(text):
    if not text.strip():
        return []
        
    # Get predictions from pipeline
    predictions = ner_pipeline(text)
    
    # Gradio HighlightedText expects a list of tuples: (text_span, label)
    # If a span has no label, we use None.
    result = []
    last_idx = 0
    
    for entity in predictions:
        start = entity['start']
        end = entity['end']
        
        # Add non-entity text
        if start > last_idx:
            result.append((text[last_idx:start], None))
            
        # Add entity text
        label = entity['entity_group']
        result.append((text[start:end], label))
        
        last_idx = end
        
    # Add remaining text
    if last_idx < len(text):
        result.append((text[last_idx:], None))
        
    return result

# 🖌️ Define color mapping
color_map = {
    "PER": "#fca5a5",  # Red
    "ORG": "#93c5fd",  # Blue
    "LOC": "#86efac",  # Green
    "MISC": "#fde047"  # Yellow
}

# 📝 Define interesting examples covering various entity types
examples = [
    ["Hugging Face Inc. is a company based in New York City. Its headquarters are in DUMBO, therefore very close to the Manhattan Bridge which is visible from the window."],
    ["My name is Sarah and I work at Google in London."],
    ["The World Health Organization warned about the new virus in Geneva."],
    ["Elon Musk founded SpaceX in Hawthorne, California."],
    ["I bought a new Apple iPhone 15 Pro yesterday from the store in Times Square."]
]

# 🏗️ Build the UI with Gradio Blocks
with gr.Blocks(title="Named Entity Recognition", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🚀 Named Entity Recognition (NER) Demo")
    gr.Markdown("Extract **Persons (PER)**, **Organizations (ORG)**, **Locations (LOC)**, and **Miscellaneous (MISC)** entities from your text.")
    
    with gr.Row():
        with gr.Column():
            input_text = gr.Textbox(
                lines=6, 
                placeholder="Enter a sentence here...", 
                label="Input Text"
            )
            submit_btn = gr.Button("Analyze Entities", variant="primary")
            
        with gr.Column():
            output_highlight = gr.HighlightedText(
                label="Extracted Entities",
                color_map=color_map,
                show_legend=True
            )
            
    # Connect logic
    submit_btn.click(fn=predict_entities, inputs=input_text, outputs=output_highlight)
    
    # Add Examples
    gr.Examples(
        examples=examples,
        inputs=input_text,
        outputs=output_highlight,
        fn=predict_entities,
        cache_examples=False
    )
    
    # Add flag button for collecting feedback (built into Blocks implicitly if we used Interface, but here we can add custom if needed)
    # Gradio recently simplified this; for true flagging we can use gr.FlaggingCallback, but a simple Markdown note suffices for demo.
    gr.Markdown("---")
    gr.Markdown("💡 *Notice incorrect predictions? In a production environment, we would include a 'Flag' button to collect user feedback for future model retraining.*")

# 🚀 Launch the app (inline in Jupyter)
demo.launch(inline=True, share=False)

## 5. How to Deploy to HuggingFace Spaces

While we can run the Gradio app locally, hosting it on the internet makes it truly accessible. **HuggingFace Spaces** offers free hosting for ML demos.

### 🌐 Step-by-Step Deployment Guide:
1. **Create an account** on [Hugging Face](https://huggingface.co/).
2. Navigate to your profile and click **New Space**.
3. Provide a name for your Space, choose an appropriate license, and select **Gradio** as the Space SDK.
4. Clone the Space repository locally or use the Web UI.
5. Upload the following files from our `deployment/` folder:
   - `app.py` (The standalone Gradio script)
   - `requirements.txt` (Containing `transformers`, `gradio`, `torch`)
   - Any necessary model files or update `app.py` to pull from the Hub.
6. Once pushed, HuggingFace Spaces will auto-build a Docker container, install dependencies from `requirements.txt`, and launch your app. It will be live for the world to see!

## 6. Project Wrap-Up 🎉

Congratulations on completing the Named Entity Recognition (NER) pipeline! Let's reflect on what we've built across these 6 notebooks.

### 🏆 What We Accomplished
1. **Exploratory Data Analysis (EDA)**: We deeply understood the CoNLL-2003 dataset, visualizing label distributions, sequence lengths, and understanding the IOB2 tagging scheme.
2. **Baseline Model**: We established a strong baseline using Conditional Random Fields (CRFs) and custom feature engineering, proving that traditional ML still holds value.
3. **Deep Learning**: We implemented a BiLSTM-CRF architecture from scratch using PyTorch, learning about embeddings, sequential modeling, and transition matrices.
4. **State-of-the-Art**: We fine-tuned a Transformer model (DistilBERT), leveraging transfer learning and attention mechanisms to achieve the highest evaluation metrics.
5. **Evaluation**: We rigorously compared our models using precision, recall, F1-score (seqeval), and confusion matrices, analyzing their strengths and failure modes.
6. **Deployment**: We finally packaged our best model into an interactive Web Application using Gradio, ready for production!

### 📊 Architecture Comparison Insights
- **CRF**: Fast to train, highly interpretable, but relies heavily on manual feature engineering. Struggles with complex semantic context.
- **BiLSTM-CRF**: Captures sequential dependencies well without manual features, but requires significant training time and data.
- **Transformers**: Absolute best performance. The pre-trained contextual embeddings provide a massive advantage, though they are computationally expensive.

### 🚀 Future Improvements
To take this project even further, one could explore:
- **Scaling Up**: Utilizing `BERT-large` or `RoBERTa` for potentially better accuracy.
- **Multilingual NER**: Fine-tuning `xlm-roberta` to recognize entities across different languages.
- **Domain Adaptation**: Fine-tuning on specialized datasets (e.g., Medical NER using BioBERT, or Financial NER).
- **MLOps**: Setting up CI/CD pipelines, model monitoring, and automated retraining.

### 🙏 Acknowledgements
- **CoNLL-2003**: For providing a foundational benchmark dataset for NER.
- **Hugging Face**: For democratizing NLP with the `transformers` and `gradio` libraries.
- **PyTorch**: For powering the deep learning ecosystem.

*Thank you for following along with this Graduation Portfolio Project!*